# Exhaustive Inference Evaluation for Incremental Fine-Tuning Analysis

This notebook provides a comprehensive inference pipeline to evaluate ProPicker checkpoints from the incremental fine-tuning experiment (EXP3). It extends previous evaluation approaches with:

1. **Multi-checkpoint evaluation**: Systematic inference on all checkpoints from increments [1, 2, 4, 8, 12, 16, 20]
2. **Multi-instance prompt evaluation**: Comparison between single-instance and 10-instance averaged prompts
3. **Comprehensive metrics**: Precision, Recall, F1, and distance-based analysis
4. **Visualization**: Performance curves and comparative analysis

## Multi-Instance Prompt Strategy

TomoTwin embeds 37×37×37 subtomograms into 32-dimensional vectors. For a **single-instance prompt**, we use one representative particle. For a **multi-instance prompt (N=10)**, we:
1. Extract N diverse particle subtomograms
2. Pass each through TomoTwin individually
3. Average the N embeddings to create a more robust prompt representation

This approach should reduce sensitivity to individual particle variations and improve generalization.

In [ ]:
import sys
from pathlib import Path

# Add project root and experiments to path
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

# Import paths (file system locations)
from paths import (
    UMU_SYNTH_DIR,
    UMU_SYNTH_TOMOS_DIR,
    UMU_SYNTH_CSV,
    TOMOTWIN_MODEL_FILE,
    PROPICKER_TOOLS_DIR,
    PROPICKER_MODEL_FILE,
    EXP3_RESULTS_DIR,
    EXP3_FINETUNING_DIR,
    EXP3_INFERENCE_DIR,
    EXP3_DATA_DIR,
    EXP3_COORDS_DIR,
    EXP3_CHECKPOINTS_DIR,
)

# Import config (experiment parameters and utilities)
from experiments.config import (
    setup_propicker_paths,
    THYROGLOBULIN_NAME,
    THYROGLOBULIN_LABEL,
    PROMPT_SIZE,
    PROMPT_HALF,
    THYROGLOBULIN_DIAMETER,
    LABEL_DIAMETER,
    EXP3_TRAIN_POOL,
    EXP3_VAL_TOMOS,
    EXP3_INCREMENTS,
)

# Setup ProPicker imports
PROPICKER_DIR = setup_propicker_paths()

# Aliases
PARTICLE_TYPE = THYROGLOBULIN_NAME
EXP_RESULTS_DIR = EXP3_RESULTS_DIR
COORDS_DIR = EXP3_COORDS_DIR

import subprocess
import json
import pandas as pd
import numpy as np
import torch
from matplotlib import pyplot as plt
from scipy import ndimage
from scipy.spatial.distance import cdist

from inference.tomotwin import get_tomotwin_prompt_embeds_dict, pass_subtomos_through_tomotwin
from utils.mrctools import load_mrc_data

import warnings
warnings.filterwarnings("ignore")

print(f"ProPicker tools: {PROPICKER_DIR}")
print(f"UMU Synth data: {UMU_SYNTH_DIR}")
print(f"TomoTwin model: {TOMOTWIN_MODEL_FILE}")
print(f"Results dir: {EXP_RESULTS_DIR}")
print(f"Checkpoints dir: {EXP3_CHECKPOINTS_DIR}")
print(f"\nExperiment parameters:")
print(f"  Validation tomograms: {EXP3_VAL_TOMOS}")
print(f"  Training increments: {EXP3_INCREMENTS}")
print(f"  Prompt size: {PROMPT_SIZE}×{PROMPT_SIZE}×{PROMPT_SIZE}")

## Step 1: Check Available Checkpoints

Verify which checkpoints from the incremental training are available for evaluation.

In [ ]:
print("=" * 70)
print("AVAILABLE CHECKPOINTS FOR EVALUATION")
print("=" * 70)

available_checkpoints = {}

for increment in EXP3_INCREMENTS:
    ckpt_path = EXP3_CHECKPOINTS_DIR / f"increment_{increment}" / "best_model.ckpt"
    if ckpt_path.exists():
        available_checkpoints[increment] = ckpt_path
        print(f"  ✅ Increment {increment:2d}: {ckpt_path.name}")
    else:
        print(f"  ❌ Increment {increment:2d}: NOT FOUND")

print(f"\n📊 Available: {len(available_checkpoints)}/{len(EXP3_INCREMENTS)} checkpoints")

if len(available_checkpoints) == 0:
    print("\n⚠️  No checkpoints found. Run the fine-tuning script first:")
    print(f"    cd {PROJECT_ROOT}/tools/ProPicker")
    print(f"    python ../../experiments/exp3_ppicker_limits/scripts/umusynth_fine_tuning.py")

## Step 2: Load Ground Truth Coordinates

Load ground truth particle coordinates for the validation set.

In [ ]:
print("=" * 70)
print("LOADING GROUND TRUTH COORDINATES")
print("=" * 70)

gt_coords_dict = {}

for tomo_name in EXP3_VAL_TOMOS:
    coord_file = COORDS_DIR / f"{tomo_name}_thyroglobulin_coords.csv"
    if coord_file.exists():
        df = pd.read_csv(coord_file)
        gt_coords_dict[tomo_name] = df[['X', 'Y', 'Z']].values
        print(f"  ✅ {tomo_name}: {len(df)} particles")
    else:
        print(f"  ❌ {tomo_name}: coords file not found")

total_particles = sum(len(coords) for coords in gt_coords_dict.values())
print(f"\n📊 Total ground truth particles: {total_particles}")

## Step 3: Extract Subtomograms for Multi-Instance Prompt

Extract multiple particle subtomograms to create both single-instance and multi-instance (10) prompts.

In [ ]:
def extract_subtomograms(tomo, coords, patch_size=37):
    """
    Extract subtomograms of given size around each coordinate.
    
    Args:
        tomo: 3D tensor (Z, Y, X)
        coords: Array of (X, Y, Z) coordinates
        patch_size: Size of subtomogram (default 37 for TomoTwin)
        
    Returns:
        List of valid subtomograms, list of valid coordinates
    """
    half = patch_size // 2
    subtomos = []
    valid_coords = []
    
    for coord in coords:
        x, y, z = coord.astype(int)
        
        # Extract patch centered at (x, y, z)
        # Note: tomo is (Z, Y, X) but coords are (X, Y, Z)
        z_start, z_end = z - half, z + half + 1
        y_start, y_end = y - half, y + half + 1
        x_start, x_end = x - half, x + half + 1
        
        # Check bounds
        if (z_start < 0 or z_end > tomo.shape[0] or
            y_start < 0 or y_end > tomo.shape[1] or
            x_start < 0 or x_end > tomo.shape[2]):
            continue
            
        subtomo = tomo[z_start:z_end, y_start:y_end, x_start:x_end]
        
        if subtomo.shape == (patch_size, patch_size, patch_size):
            subtomos.append(subtomo)
            valid_coords.append(coord)
            
    return subtomos, np.array(valid_coords) if valid_coords else np.array([]).reshape(0, 3)

# Extract subtomograms from TRAINING tomograms (not validation)
# to avoid data leakage
print("=" * 70)
print("EXTRACTING SUBTOMOGRAMS FOR PROMPT GENERATION")
print("=" * 70)

all_subtomos = []
all_coords = []

# Use first few training tomograms to get diverse samples
source_tomos = EXP3_TRAIN_POOL[:5]  # Use 5 training tomos

for tomo_name in source_tomos:
    # Load tomogram
    tomo_path = UMU_SYNTH_TOMOS_DIR / f"{tomo_name}.mrc"
    if not tomo_path.exists():
        continue
    
    tomo = load_mrc_data(str(tomo_path)).float()
    
    # Load coordinates
    coord_file = COORDS_DIR / f"{tomo_name}_thyroglobulin_coords.csv"
    if not coord_file.exists():
        continue
    
    coords_df = pd.read_csv(coord_file)
    coords = coords_df[['X', 'Y', 'Z']].values
    
    # Extract subtomograms
    subtomos, valid_coords = extract_subtomograms(tomo, coords, patch_size=PROMPT_SIZE)
    
    all_subtomos.extend(subtomos)
    all_coords.extend(valid_coords)
    
    print(f"  {tomo_name}: extracted {len(subtomos)} subtomograms")
    del tomo

print(f"\n📊 Total extracted subtomograms: {len(all_subtomos)}")

In [ ]:
# Visualize some extracted subtomograms
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    if i < len(all_subtomos):
        subtomo = all_subtomos[i]
        if isinstance(subtomo, torch.Tensor):
            subtomo = subtomo.numpy()
        central_slice = subtomo[PROMPT_SIZE // 2]
        ax.imshow(central_slice, cmap='gray')
        ax.set_title(f"Particle #{i+1}")
    ax.axis('off')

plt.suptitle(f"Sample extracted subtomograms ({PROMPT_SIZE}×{PROMPT_SIZE}×{PROMPT_SIZE})")
plt.tight_layout()
plt.show()

## Step 4: Generate Single and Multi-Instance Prompts

Create two types of prompt embeddings:
1. **Single-instance prompt**: One representative particle (same as original fine-tuning)
2. **Multi-instance prompt (N=10)**: Average of 10 diverse particle embeddings

In [ ]:
def select_diverse_samples(subtomos, n_samples=10, method='variance'):
    """
    Select N diverse subtomograms from a larger set.
    
    Args:
        subtomos: List of subtomograms
        n_samples: Number of samples to select
        method: Selection method ('variance', 'random', 'uniform')
    
    Returns:
        List of selected subtomogram indices
    """
    if len(subtomos) <= n_samples:
        return list(range(len(subtomos)))
    
    if method == 'random':
        return list(np.random.choice(len(subtomos), n_samples, replace=False))
    
    elif method == 'uniform':
        # Uniformly spaced indices
        return list(np.linspace(0, len(subtomos)-1, n_samples, dtype=int))
    
    elif method == 'variance':
        # Select based on variance ranking (diverse variance levels)
        variances = []
        for s in subtomos:
            if isinstance(s, torch.Tensor):
                s = s.numpy()
            variances.append(np.var(s))
        
        # Sort by variance and take uniformly from the ranked list
        sorted_indices = np.argsort(variances)
        selected = sorted_indices[np.linspace(0, len(sorted_indices)-1, n_samples, dtype=int)]
        return list(selected)
    
    return list(range(n_samples))

# Number of instances for multi-prompt
N_INSTANCES = 10

print("=" * 70)
print("GENERATING PROMPT EMBEDDINGS")
print("=" * 70)

# Convert subtomograms to tensor
subtomos_tensor = torch.stack([torch.tensor(s) if not isinstance(s, torch.Tensor) else s 
                               for s in all_subtomos])

print(f"\nSubtomograms tensor shape: {subtomos_tensor.shape}")

# Select diverse samples for multi-instance prompt
selected_indices = select_diverse_samples(all_subtomos, n_samples=N_INSTANCES, method='variance')
print(f"\nSelected indices for {N_INSTANCES}-instance prompt: {selected_indices}")

# Generate embeddings for all selected subtomograms
print(f"\nGenerating embeddings using TomoTwin...")
selected_subtomos = subtomos_tensor[selected_indices]

# Pass through TomoTwin
embeddings = pass_subtomos_through_tomotwin(
    subtomos=selected_subtomos,
    tomotwin_model_file=str(TOMOTWIN_MODEL_FILE),
    batch_size=N_INSTANCES,
    device="cuda:0" if torch.cuda.is_available() else "cpu"
)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

In [ ]:
# Create single-instance prompt (first selected sample)
single_prompt_embedding = embeddings[0]

# Create multi-instance prompt (average of N embeddings)
multi_prompt_embedding = embeddings.mean(dim=0)

print("=" * 70)
print("PROMPT EMBEDDINGS COMPARISON")
print("=" * 70)

print(f"\n📌 Single-instance prompt:")
print(f"   Shape: {single_prompt_embedding.shape}")
print(f"   Norm: {single_prompt_embedding.norm():.4f}")
print(f"   Mean: {single_prompt_embedding.mean():.4f}")
print(f"   Std: {single_prompt_embedding.std():.4f}")

print(f"\n📌 Multi-instance prompt (N={N_INSTANCES}):")
print(f"   Shape: {multi_prompt_embedding.shape}")
print(f"   Norm: {multi_prompt_embedding.norm():.4f}")
print(f"   Mean: {multi_prompt_embedding.mean():.4f}")
print(f"   Std: {multi_prompt_embedding.std():.4f}")

# Cosine similarity between single and multi
cos_sim = torch.nn.functional.cosine_similarity(
    single_prompt_embedding.unsqueeze(0), 
    multi_prompt_embedding.unsqueeze(0)
).item()
print(f"\n📊 Cosine similarity (single vs multi): {cos_sim:.4f}")

# Variance of individual embeddings
embedding_variances = embeddings.var(dim=0).mean().item()
print(f"📊 Mean variance across embedding dimensions: {embedding_variances:.6f}")

In [ ]:
# Save prompt embeddings to files
PROMPTS_DIR = EXP_RESULTS_DIR / "prompts"
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

# Single-instance prompt
single_prompt_file = PROMPTS_DIR / "single_instance_prompt.json"
single_prompt_dict = {PARTICLE_TYPE: single_prompt_embedding.tolist()}
with open(single_prompt_file, 'w') as f:
    json.dump(single_prompt_dict, f, indent=4)
print(f"✅ Single-instance prompt saved to: {single_prompt_file}")

# Multi-instance prompt
multi_prompt_file = PROMPTS_DIR / f"multi_instance_prompt_n{N_INSTANCES}.json"
multi_prompt_dict = {PARTICLE_TYPE: multi_prompt_embedding.tolist()}
with open(multi_prompt_file, 'w') as f:
    json.dump(multi_prompt_dict, f, indent=4)
print(f"✅ Multi-instance prompt (N={N_INSTANCES}) saved to: {multi_prompt_file}")

# Also save all individual embeddings for analysis
all_embeddings_file = PROMPTS_DIR / f"all_embeddings_n{N_INSTANCES}.pt"
torch.save({
    'embeddings': embeddings,
    'indices': selected_indices,
    'single_prompt': single_prompt_embedding,
    'multi_prompt': multi_prompt_embedding,
}, all_embeddings_file)
print(f"✅ All embeddings saved to: {all_embeddings_file}")

In [ ]:
# Visualize embedding space
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PCA of embeddings
pca = PCA(n_components=2)
embeddings_np = embeddings.numpy()
embeddings_2d = pca.fit_transform(embeddings_np)
multi_2d = pca.transform(multi_prompt_embedding.unsqueeze(0).numpy())

ax1 = axes[0]
ax1.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c='steelblue', s=100, alpha=0.7, label='Individual')
ax1.scatter(multi_2d[:, 0], multi_2d[:, 1], c='red', s=200, marker='*', label='Average (Multi)', zorder=5)
ax1.scatter(embeddings_2d[0, 0], embeddings_2d[0, 1], c='green', s=150, marker='s', label='Single', zorder=5)

for i, (x, y) in enumerate(embeddings_2d):
    ax1.annotate(str(i+1), (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)

ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title(f'Embedding Space (PCA, {pca.explained_variance_ratio_.sum()*100:.1f}% variance)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Embedding dimension heatmap
ax2 = axes[1]
im = ax2.imshow(embeddings_np, aspect='auto', cmap='RdBu_r')
ax2.set_xlabel('Embedding Dimension')
ax2.set_ylabel('Sample Index')
ax2.set_title('Individual Embeddings Heatmap')
plt.colorbar(im, ax=ax2, label='Value')

plt.tight_layout()
plt.savefig(EXP_RESULTS_DIR / "prompt_embeddings_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Analysis plot saved to: {EXP_RESULTS_DIR / 'prompt_embeddings_analysis.png'}")

## Step 5: Evaluation Functions

Define functions to evaluate predictions against ground truth.

In [ ]:
def evaluate_predictions(pred_coords, gt_coords, distance_thresh=18.5):
    """
    Evaluate predicted coordinates against ground truth.
    
    Args:
        pred_coords: Predicted coordinates (N, 3)
        gt_coords: Ground truth coordinates (M, 3)
        distance_thresh: Maximum distance for a match (default: PROMPT_SIZE/2)
    
    Returns:
        Dictionary with evaluation metrics
    """
    n_gt = len(gt_coords)
    n_pred = len(pred_coords)
    
    matched_gt = set()
    matched_pred = set()
    
    if n_pred > 0 and n_gt > 0:
        distances = cdist(pred_coords, gt_coords)
        
        # Greedy matching
        for i in range(len(pred_coords)):
            min_dist_idx = np.argmin(distances[i])
            if distances[i, min_dist_idx] < distance_thresh and min_dist_idx not in matched_gt:
                matched_gt.add(min_dist_idx)
                matched_pred.add(i)
    
    tp = len(matched_gt)
    fp = n_pred - tp
    fn = n_gt - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'n_gt': n_gt,
        'n_pred': n_pred,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


def load_predicted_coords(coords_file):
    """
    Load predicted coordinates from a .coords file.
    """
    if not coords_file.exists():
        return np.array([]).reshape(0, 3)
    
    df = pd.read_csv(coords_file, sep='\t', header=None, names=['X', 'Y', 'Z'])
    return df[['X', 'Y', 'Z']].values


def extract_coords_from_locmap(locmap_file, threshold=0.5):
    """
    Extract particle coordinates from a localization map.
    """
    locmap_data = torch.load(locmap_file, weights_only=False)
    
    if isinstance(locmap_data, dict):
        pred_locmap = list(locmap_data.values())[0]
        if hasattr(pred_locmap, 'numpy'):
            pred_locmap = pred_locmap.numpy()
    elif hasattr(locmap_data, 'numpy'):
        pred_locmap = locmap_data.numpy()
    else:
        pred_locmap = locmap_data
    
    # Handle 4D (C, D, H, W) format
    if len(pred_locmap.shape) == 4:
        if pred_locmap.shape[0] > 1:
            pred_locmap = pred_locmap[1]  # Foreground channel
        else:
            pred_locmap = pred_locmap[0]
    
    # Binarization and clustering
    binary_locmap = pred_locmap > threshold
    labeled_array, num_features = ndimage.label(binary_locmap)
    
    if num_features > 0:
        centroids = ndimage.center_of_mass(binary_locmap, labeled_array, range(1, num_features + 1))
        pred_coords = np.array(centroids)[:, ::-1]  # Convert (Z, Y, X) to (X, Y, Z)
    else:
        pred_coords = np.array([]).reshape(0, 3)
    
    return pred_coords

print("✅ Evaluation functions defined")

## Step 6: Evaluate All Checkpoints

Run inference and evaluation for each available checkpoint.

In [ ]:
print("=" * 70)
print("EVALUATING CHECKPOINTS FROM INFERENCE RESULTS")
print("=" * 70)

# Collect results for each increment
all_results = []

for increment in EXP3_INCREMENTS:
    inference_dir = EXP3_INFERENCE_DIR / f"increment_{increment}"
    
    if not inference_dir.exists():
        print(f"\n❌ Increment {increment}: No inference results found")
        continue
    
    print(f"\n📊 Evaluating Increment {increment}...")
    
    # Try to load from Coords_All first, then from locmaps
    coords_dir = inference_dir / "Coords_All"
    locmaps_dir = inference_dir / "locmaps"
    
    increment_results = []
    
    for tomo_name in EXP3_VAL_TOMOS:
        gt_coords = gt_coords_dict.get(tomo_name)
        if gt_coords is None:
            continue
        
        # Load predictions
        pred_coords = None
        
        if coords_dir.exists():
            coords_file = coords_dir / f"{tomo_name}.coords"
            if coords_file.exists():
                pred_coords = load_predicted_coords(coords_file)
        
        if pred_coords is None and locmaps_dir.exists():
            locmap_file = locmaps_dir / f"{tomo_name}.pt"
            if locmap_file.exists():
                pred_coords = extract_coords_from_locmap(locmap_file)
        
        if pred_coords is None:
            print(f"   ⚠️ {tomo_name}: No predictions found")
            continue
        
        # Evaluate
        metrics = evaluate_predictions(pred_coords, gt_coords, distance_thresh=PROMPT_SIZE/2)
        metrics['tomo'] = tomo_name
        metrics['increment'] = increment
        increment_results.append(metrics)
        
        print(f"   {tomo_name}: P={metrics['precision']:.3f}, R={metrics['recall']:.3f}, F1={metrics['f1']:.3f}")
    
    if increment_results:
        all_results.extend(increment_results)
        
        # Summary for this increment
        inc_df = pd.DataFrame(increment_results)
        print(f"   → Mean: P={inc_df['precision'].mean():.3f}, R={inc_df['recall'].mean():.3f}, F1={inc_df['f1'].mean():.3f}")

if all_results:
    results_df = pd.DataFrame(all_results)
    print(f"\n✅ Evaluated {len(results_df)} tomogram-checkpoint combinations")

In [ ]:
# Summary statistics per increment
if all_results:
    print("=" * 70)
    print("SUMMARY: PERFORMANCE BY TRAINING INCREMENT")
    print("=" * 70)
    
    summary_df = results_df.groupby('increment').agg({
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std'],
        'f1': ['mean', 'std'],
        'tp': 'sum',
        'fp': 'sum',
        'fn': 'sum',
    }).round(3)
    
    summary_df.columns = ['_'.join(col).strip() for col in summary_df.columns.values]
    print(summary_df.to_string())
    
    # Save results
    results_file = EXP_RESULTS_DIR / "inference_evaluation_results.csv"
    results_df.to_csv(results_file, index=False)
    print(f"\n✅ Detailed results saved to: {results_file}")
    
    summary_file = EXP_RESULTS_DIR / "inference_evaluation_summary.csv"
    summary_df.to_csv(summary_file)
    print(f"✅ Summary saved to: {summary_file}")

In [ ]:
# Visualization: Performance vs Training Data
if all_results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Aggregate by increment
    agg_df = results_df.groupby('increment').agg({
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std'],
        'f1': ['mean', 'std'],
    }).reset_index()
    
    increments = agg_df['increment'].values
    
    # Precision
    ax1 = axes[0]
    ax1.errorbar(increments, agg_df[('precision', 'mean')], 
                 yerr=agg_df[('precision', 'std')], 
                 marker='o', capsize=5, linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Training Tomograms')
    ax1.set_ylabel('Precision')
    ax1.set_title('Precision vs Training Data')
    ax1.set_xticks(EXP3_INCREMENTS)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.05)
    
    # Recall
    ax2 = axes[1]
    ax2.errorbar(increments, agg_df[('recall', 'mean')], 
                 yerr=agg_df[('recall', 'std')], 
                 marker='s', capsize=5, linewidth=2, markersize=8, color='orange')
    ax2.set_xlabel('Number of Training Tomograms')
    ax2.set_ylabel('Recall')
    ax2.set_title('Recall vs Training Data')
    ax2.set_xticks(EXP3_INCREMENTS)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1.05)
    
    # F1
    ax3 = axes[2]
    ax3.errorbar(increments, agg_df[('f1', 'mean')], 
                 yerr=agg_df[('f1', 'std')], 
                 marker='^', capsize=5, linewidth=2, markersize=8, color='green')
    ax3.set_xlabel('Number of Training Tomograms')
    ax3.set_ylabel('F1 Score')
    ax3.set_title('F1 Score vs Training Data')
    ax3.set_xticks(EXP3_INCREMENTS)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.savefig(EXP_RESULTS_DIR / "performance_vs_training_data.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Performance plot saved to: {EXP_RESULTS_DIR / 'performance_vs_training_data.png'}")

## Step 7: Compare Single vs Multi-Instance Prompts

To properly compare prompt strategies, we need to run inference with both prompt types. This section sets up the comparison and provides instructions for running the inference scripts.

In [ ]:
print("=" * 70)
print("PROMPT COMPARISON SETUP")
print("=" * 70)

print(f"\n📁 Generated prompt files:")
print(f"   Single-instance: {single_prompt_file}")
print(f"   Multi-instance:  {multi_prompt_file}")

print(f"\n📋 To run inference with different prompts:")
print(f"")
print(f"   # Navigate to ProPicker directory")
print(f"   cd {PROJECT_ROOT}/tools/ProPicker")
print(f"")
print(f"   # Run inference with single-instance prompt")
print(f"   python ../../experiments/exp3_ppicker_limits/scripts/umusynth_inference.py \\")
print(f"       --prompt-file {single_prompt_file}")
print(f"")
print(f"   # Run inference with multi-instance prompt")
print(f"   python ../../experiments/exp3_ppicker_limits/scripts/umusynth_inference.py \\")
print(f"       --prompt-file {multi_prompt_file}")
print(f"")
print(f"Note: The inference script needs to be modified to accept a custom prompt file.")
print(f"Alternatively, manually copy the prompt file to the expected location before running.")

In [ ]:
# Check if we have results from both prompt types
single_results_dir = EXP3_INFERENCE_DIR / "single_prompt"
multi_results_dir = EXP3_INFERENCE_DIR / f"multi_prompt_n{N_INSTANCES}"

has_single = single_results_dir.exists() and any(single_results_dir.iterdir())
has_multi = multi_results_dir.exists() and any(multi_results_dir.iterdir())

if has_single and has_multi:
    print("✅ Both prompt type results available for comparison")
    
    # Evaluate both
    comparison_results = []
    
    for prompt_type, results_dir in [('single', single_results_dir), ('multi', multi_results_dir)]:
        for increment in EXP3_INCREMENTS:
            inc_dir = results_dir / f"increment_{increment}"
            if not inc_dir.exists():
                continue
            
            coords_dir = inc_dir / "Coords_All"
            
            for tomo_name in EXP3_VAL_TOMOS:
                gt_coords = gt_coords_dict.get(tomo_name)
                if gt_coords is None:
                    continue
                
                coords_file = coords_dir / f"{tomo_name}.coords"
                if coords_file.exists():
                    pred_coords = load_predicted_coords(coords_file)
                    metrics = evaluate_predictions(pred_coords, gt_coords)
                    metrics['tomo'] = tomo_name
                    metrics['increment'] = increment
                    metrics['prompt_type'] = prompt_type
                    comparison_results.append(metrics)
    
    if comparison_results:
        comp_df = pd.DataFrame(comparison_results)
        
        # Summary comparison
        print("\n" + "=" * 70)
        print("SINGLE vs MULTI-INSTANCE PROMPT COMPARISON")
        print("=" * 70)
        
        comp_summary = comp_df.groupby(['prompt_type', 'increment']).agg({
            'f1': 'mean',
            'precision': 'mean',
            'recall': 'mean',
        }).round(3)
        
        print(comp_summary.to_string())
else:
    print("⚠️ Prompt comparison results not yet available.")
    print(f"   Run inference with both prompt types first.")
    print(f"   Expected directories:")
    print(f"   - {single_results_dir}")
    print(f"   - {multi_results_dir}")

## Step 8: Visualization of Inference Results

In [ ]:
# Visualize localization maps for different increments
def visualize_locmaps_comparison(tomo_name, increments_to_show=[1, 4, 20]):
    """
    Visualize localization maps from different training increments for comparison.
    """
    # Load original tomogram
    tomo_file = UMU_SYNTH_TOMOS_DIR / f"{tomo_name}.mrc"
    if not tomo_file.exists():
        print(f"⚠️ Tomogram not found: {tomo_file}")
        return
    
    tomo = load_mrc_data(str(tomo_file)).numpy()
    
    # Get ground truth
    gt_coords = gt_coords_dict.get(tomo_name, np.array([]).reshape(0, 3))
    
    # Find mid-z slice with particles
    if len(gt_coords) > 0:
        mid_z = int(np.median(gt_coords[:, 2]))
    else:
        mid_z = tomo.shape[0] // 2
    
    n_plots = len(increments_to_show) + 1
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))
    
    # Original tomogram
    axes[0].imshow(tomo[mid_z], cmap='gray')
    if len(gt_coords) > 0:
        z_mask = np.abs(gt_coords[:, 2] - mid_z) < 10
        axes[0].scatter(gt_coords[z_mask, 0], gt_coords[z_mask, 1], 
                       c='lime', s=50, marker='o', facecolors='none', linewidths=2)
    axes[0].set_title(f'Original (z={mid_z})\n{len(gt_coords)} GT particles')
    axes[0].axis('off')
    
    # Localization maps for each increment
    for i, increment in enumerate(increments_to_show):
        ax = axes[i + 1]
        
        locmap_file = EXP3_INFERENCE_DIR / f"increment_{increment}" / "locmaps" / f"{tomo_name}.pt"
        
        if locmap_file.exists():
            locmap_data = torch.load(locmap_file, weights_only=False)
            
            if isinstance(locmap_data, dict):
                pred_locmap = list(locmap_data.values())[0]
                if hasattr(pred_locmap, 'numpy'):
                    pred_locmap = pred_locmap.numpy()
            elif hasattr(locmap_data, 'numpy'):
                pred_locmap = locmap_data.numpy()
            else:
                pred_locmap = locmap_data
            
            if len(pred_locmap.shape) == 4:
                pred_locmap = pred_locmap[1] if pred_locmap.shape[0] > 1 else pred_locmap[0]
            
            ax.imshow(tomo[mid_z], cmap='gray')
            ax.imshow(pred_locmap[mid_z], cmap='hot', alpha=0.6)
            ax.set_title(f'Increment {increment} tomos')
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Increment {increment} (N/A)')
        
        ax.axis('off')
    
    plt.suptitle(f'Localization Maps Comparison: {tomo_name}')
    plt.tight_layout()
    return fig

# Visualize for first validation tomogram
if EXP3_VAL_TOMOS:
    fig = visualize_locmaps_comparison(EXP3_VAL_TOMOS[0], increments_to_show=[1, 4, 8, 20])
    if fig:
        plt.savefig(EXP_RESULTS_DIR / f"locmaps_comparison_{EXP3_VAL_TOMOS[0]}.png", 
                    dpi=150, bbox_inches='tight')
        plt.show()
        print(f"\n✅ Comparison plot saved")

## Summary and Conclusions

This notebook provides a comprehensive evaluation pipeline for the incremental fine-tuning experiment. Key findings:

In [ ]:
if all_results:
    print("=" * 70)
    print("EXPERIMENT SUMMARY")
    print("=" * 70)
    
    # Find optimal increment
    summary_by_inc = results_df.groupby('increment')['f1'].mean()
    best_increment = summary_by_inc.idxmax()
    best_f1 = summary_by_inc.max()
    
    print(f"\n🏆 Best performing increment: {best_increment} tomograms (F1={best_f1:.3f})")
    
    # Find minimum viable increment (90% of best performance)
    threshold_f1 = 0.9 * best_f1
    viable_increments = summary_by_inc[summary_by_inc >= threshold_f1].index.tolist()
    min_viable = min(viable_increments) if viable_increments else None
    
    if min_viable:
        print(f"📊 Minimum viable increment (≥90% of best): {min_viable} tomograms")
        print(f"   → Data efficiency: achieve {summary_by_inc[min_viable]/best_f1*100:.1f}% of best with {min_viable/best_increment*100:.1f}% of data")
    
    # Performance plateau analysis
    improvements = summary_by_inc.diff()
    plateau_increment = None
    for inc in sorted(summary_by_inc.index)[1:]:
        if improvements[inc] < 0.01:  # Less than 1% improvement
            plateau_increment = inc
            break
    
    if plateau_increment:
        print(f"📈 Performance plateau reached at: {plateau_increment} tomograms")
    
    print(f"\n📋 Full results saved to: {EXP_RESULTS_DIR}")
else:
    print("⚠️ No evaluation results available.")
    print("   Run the fine-tuning and inference scripts first.")

## Next Steps

1. **Run remaining training increments** if any checkpoints are missing
2. **Run inference** with both single and multi-instance prompts for comprehensive comparison
3. **Analyze prompt sensitivity** to understand when multi-instance prompts provide benefit
4. **Document findings** for the final report